# 1 - Preprocessing

Parses the EyeLink `.asc` files of one experiment into the long-format table
every later notebook reads: one row per saccade, carrying the properties of
both sounds in the trial.

Set `EXPERIMENT` in the configuration cell and run the notebook once per
experiment: `EX_1` (Exp 1a), `EX_9` (Exp 1b), `EX_7` (Exp 2), `EX_3` (Exp 3).

**Inputs**  `data_raw/{EXPERIMENT}/asc/*.asc` (+ `csv/beh_*.csv` for EX_1 and EX_9)  
**Output**  `data_preprocessed/{EXPERIMENT}/long_format_alt_all.csv`

> The aggregation step globs *every* `*_long_alt.csv` in `long_format_alt/`,
> whether or not the subject's raw `.asc` is still there. After
> re-preprocessing, check the subject count in the validation cell against the
> number of `.asc` files.


In [ ]:
%load_ext autoreload
%autoreload 2

# === Import libraries ===
import pandas as pd
import numpy as np
import csv
import os
import matplotlib.pyplot as plt

# Import standard preprocessing functions (for parsing)
from src.preprocessing import (
    find_experiment_data,
    find_experiment_data_by_trial_number,
    extract_sound_events,
    extract_answer_events,
    find_rt,
    parse_eblink,
    parse_efix,
    parse_esacc
)

# Import alternative preprocessing functions
from src.preprocessing_alt import (
    generate_long_format_alt,
    collect_all_long_files_alt,
    extract_trial_boundaries_from_trial_markers
)

In [ ]:
# ====================================================================
# EXPERIMENT CONFIGURATION
# ====================================================================

EXPERIMENT_CONFIGS = {
    # Directory names are historical; the paper numbering is in the comments.
    'EX_1': {   # Experiment 1a - binaural-direction, 10 ms pink noise
        'name': 'EX_1',
        'description': 'Sound localization (left/right direction)',
        'sound_markers': ['FIRST_SOUND_PLAYED', 'SECOND_SOUND_PLAYED'],
        'answer_markers': ['ANSWER_LEFT', 'ANSWER_RIGHT'],
        'uses_behavioral_csv': True,
        'has_pitch': False,
        'target_field': 'sound_direction',
        'answer_mapping': lambda x: x.replace('ANSWER_', '').lower(),
        'training_removal': 'image_based',
    },
    'EX_9': {   # Experiment 1b - binaural-direction, 25 ms pink noise
        'name': 'EX_9',
        'description': 'Sound localization (left/right direction), longer stimulus',
        'sound_markers': ['FIRST_SOUND_PLAYED', 'SECOND_SOUND_PLAYED'],
        'answer_markers': ['ANSWER_LEFT', 'ANSWER_RIGHT'],
        'uses_behavioral_csv': True,
        'has_pitch': False,
        'target_field': 'sound_direction',
        'answer_mapping': lambda x: x.replace('ANSWER_', '').lower(),
        'training_removal': 'image_based',
    },
    'EX_7': {   # Experiment 2 - monaural-direction, 6 ms tone
        'name': 'EX_7',
        'description': 'Sound localization (left/right direction) with pitch-varying stimuli',
        'sound_markers': ['PLAYED_SOUND_LEFT_HIGH', 'PLAYED_SOUND_RIGHT_HIGH',
                          'PLAYED_SOUND_LEFT_LOW', 'PLAYED_SOUND_RIGHT_LOW',
                          'PLAYED_SOUND_EMPTY'],
        'answer_markers': ['ANSWER_LEFT', 'ANSWER_RIGHT'],
        'uses_behavioral_csv': False,
        'has_pitch': True,
        'target_field': 'sound_direction',   # pitch varies but is task-irrelevant
        'answer_mapping': lambda x: x.replace('ANSWER_', '').lower(),
        'consistency_note': 'Task is sound localization, but sounds also vary in pitch (irrelevant to task)',
        'training_removal': 'image_based',
    },
    'EX_3': {   # Experiment 3 - monaural-pitch, 6 ms tone
        'name': 'EX_3',
        'description': 'Pitch discrimination (high/low pitch)',
        'sound_markers': ['PLAYED_SOUND_LEFT_HIGH', 'PLAYED_SOUND_RIGHT_HIGH',
                          'PLAYED_SOUND_LEFT_LOW', 'PLAYED_SOUND_RIGHT_LOW',
                          'PLAYED_SOUND_EMPTY'],
        # Answered on the up/down arrows, not left/right: 'high' -> up, 'low' -> down.
        'answer_markers': ['ANSWER_UP', 'ANSWER_DOWN'],
        'uses_behavioral_csv': False,
        'has_pitch': True,
        'target_field': 'sound_pitch',
        'answer_mapping': lambda x: 'high' if 'UP' in x else 'low',
        'consistency_note': 'Task is pitch discrimination, but sounds still have spatial location (left/right ear)',
        'training_removal': 'image_based',
    },
}


# === SELECT EXPERIMENT ===
EXPERIMENT = 'EX_1'  # 'EX_1' (Exp 1a), 'EX_9' (Exp 1b), 'EX_7' (Exp 2), 'EX_3' (Exp 3)
config = EXPERIMENT_CONFIGS[EXPERIMENT]

print(f"Processing {config['name']}: {config['description']}")
print(f"Target field for accuracy: {config['target_field']}")
print(f"Training removal method: {config['training_removal']}")
print("\n=== ALTERNATIVE PREPROCESSING ===")
print("This notebook uses trial-based preprocessing:")
print("- All saccades within trial boundaries (IMAGE_ONSET to IMAGE_OFFSET) are included")
print("- No search window filtering")
print("- No duration/amplitude filtering")
print("- Each row contains information about BOTH sounds in the trial")
print("- Timing calculated relative to sound 1, sound 2, and closest sound")

# === PROCESSING PARAMETERS ===
et_directory = f'data_raw/{EXPERIMENT}/asc/'
csv_directory = f"data_raw/{EXPERIMENT}/csv/"

# Eye selection: "L" for left eye, "R" for right eye, None for both eyes
eye = "R"

# Output directory
os.makedirs(f'data_preprocessed/{EXPERIMENT}', exist_ok=True)

In [ ]:
# ====================================================================
# MAIN PROCESSING - ALTERNATIVE APPROACH
# ====================================================================

# === Process data for each file ===
all_files_et = [f for f in os.listdir(et_directory) if f.endswith('.asc')]
total_files = len(all_files_et)

print(f"Processing {total_files} files\n")

# for idx, file in enumerate(all_files_et[:1], start=1):  # only first file for testing
# for idx, file in enumerate(all_files_et[25:], start=1):  
for idx, file in enumerate(all_files_et, start=1): # all
    id = file.split('.')[0]
    print(f'\nProcessing: {id} ({idx}/{total_files})')

    # Load behavioral CSV if needed
    behavior_df = None
    if config['uses_behavioral_csv']:
        csv_file = [f for f in os.listdir(csv_directory) if f.startswith(f"beh_{id}")]
        if not csv_file:
            print(f"CSV for {id} not found!")
            continue
        behavior_df = pd.read_csv(os.path.join(csv_directory, csv_file[0]), encoding='latin1')

    # Load eye-tracking data
    et_data = []
    with open(et_directory + file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            
            # TRIAL markers can contain brackets and spaces, e.g.
            # "MSG 3528862 TRIAL_0_[['SSACC', 30], 'left_low']_START"
            # which splits incorrectly, so merge from element 2 onwards
            # BUT NOT TRIAL_RESULT (which is used by find_experiment_data)
            if (len(parts) > 3 and 
                parts[0] == 'MSG' and 
                parts[2].startswith('TRIAL_') and
                'RESULT' not in parts[2]):
                merged = ' '.join(parts[2:])
                parts = parts[:2] + [merged]
            
            et_data.append(parts)

    ## === Remove training data (method depends on experiment) ===
    if config['training_removal'] == 'image_based':
        et_data = find_experiment_data(et_data)
    elif config['training_removal'] == 'trial_number':
        min_trial = config.get('min_trial_number', 20)
        et_data = find_experiment_data_by_trial_number(et_data, min_trial_number=min_trial)

    ## === Extract sounds, responses and combine into one dataset (matched) ===
    sound_events = extract_sound_events(et_data, behavior_df, config)
    answer_events = extract_answer_events(et_data, config)
    matched = find_rt(sound_events, answer_events)

    # === Extract blink data ===
    blinks = parse_eblink(et_data, eye=eye)
    print(f"Found {len(blinks)} blinks")

    # === Extract ALL saccades (no filtering) ===
    # Note: We pass max_duration=None and min_x_distance=None to disable filtering
    saccades, statistics = parse_esacc(et_data, blinks=blinks, eye=eye, max_duration=None, min_x_distance=None)
    print(f"Found {len(saccades)} saccades (unfiltered)")

    # === Extract fixation data ===
    fixations = parse_efix(et_data, eye=eye)
    print(f"Found {len(fixations)} fixations")

    # == Generate alternative long format ==
    # This uses trial boundaries (IMAGE_ONSET to IMAGE_OFFSET) and includes all saccades
    generate_long_format_alt(id, saccades, matched, config, et_data, fixations=fixations)


# === Save collected file ===
print("\n" + "="*70)
print("COLLECTING ALL INDIVIDUAL FILES")
print("="*70)
collect_all_long_files_alt(config)

In [ ]:
# ====================================================================
# DATA VALIDATION
# ====================================================================

# Load the aggregated file
df_all = pd.read_csv(f'data_preprocessed/{EXPERIMENT}/long_format_alt_all.csv',
                     low_memory=False)

print("=== DATA VALIDATION ===")
print(f"Total rows: {len(df_all):,}")
print(f"Total subjects: {df_all['id_subject'].nunique()}")
print(f"Total trials: {df_all['trial_number'].nunique()}")
print(f"Trials per subject:")
print(df_all.groupby('id_subject')['trial_number'].nunique().describe())

print(f"Saccades per trial (including trials with 0 saccades):")
print(df_all.groupby(['id_subject', 'trial_number'])['id_sacc'].count().describe())

print(f"Trials with no saccades: {df_all['id_sacc'].isna().sum()}")
print(f"Trials with saccades: {df_all['id_sacc'].notna().sum()}")

print(f"Closest sound distribution:")
print(df_all['closest_sound'].value_counts())

print(f"Consistency with closest sound:")
print(df_all['consistent_with_closest'].value_counts(dropna=False))

# --- Sound direction (if present) ---
if 'sound_1_direction' in df_all.columns or 'sound_2_direction' in df_all.columns:
    df_trials = df_all.drop_duplicates(subset=['id_subject', 'trial_number'])
    dir_records = []
    for sound in ['sound_1', 'sound_2']:
        col = f'{sound}_direction'
        if col in df_trials.columns:
            tmp = df_trials[['id_subject', col]].copy()
            tmp.columns = ['id_subject', 'direction']
            dir_records.append(tmp)
    if dir_records:
        df_dir = pd.concat(dir_records, ignore_index=True).dropna(subset=['direction'])
        print(f"Sound direction counts per subject (sound 1 + sound 2 combined):")
        print(df_dir.groupby(['id_subject', 'direction']).size().unstack(fill_value=0).to_string)

# --- Sound pitch (if present) ---
if 'sound_1_pitch' in df_all.columns or 'sound_2_pitch' in df_all.columns:
    df_trials = df_all.drop_duplicates(subset=['id_subject', 'trial_number'])
    pitch_records = []
    for sound in ['sound_1', 'sound_2']:
        col = f'{sound}_pitch'
        if col in df_trials.columns:
            tmp = df_trials[['id_subject', col]].copy()
            tmp.columns = ['id_subject', 'pitch']
            pitch_records.append(tmp)
    if pitch_records:
        df_pitch = pd.concat(pitch_records, ignore_index=True).dropna(subset=['pitch'])
        print(f"Sound pitch counts per subject (sound 1 + sound 2 combined):")
        print(df_pitch.groupby(['id_subject', 'pitch']).size().unstack(fill_value=0).to_string())
